# Syria Displacement and Return Atlas (2026)

Overview

This notebook creates an interactive displacement and return atlas for Syria using the IOM displacement and return baseline dataset for 2026.

The IOM dataset contains population indicators for reported locations but does not provide point geometries. To create a spatial representation, IOM locations are matched to HOTOSM populated-place points within the same ADM3 area using normalised settlement names and text-similarity scores.

Only matches that satisfy the defined acceptance criteria are included in the primary population layers. Matching candidates, similarity scores, acceptance decisions and exclusions are retained as audit data.

このNotebookでは、IOMの2026年避難・帰還ベースラインデータを使用し、シリアの避難・帰還状況をインタラクティブ地図として表示します。

IOMデータには報告地点ごとの人口指標が含まれていますが、地点ジオメトリは含まれていません。そのため、同一ADM3内にあるHOTOSMの集落ポイントと、正規化した地名および文字列類似度を用いて照合します。

定義した採用基準を満たす照合結果のみを主要な人口レイヤーに使用します。候補地点、類似度、採用判定および除外理由は、監査用データとして保存します。

Objectives

- Read and validate the IOM and HOTOSM source datasets
- Standardise administrative and population-indicator fields
- Validate population values and source totals
- Match IOM locations to HOTOSM populated-place points within common ADM3 areas
- Prevent multiple IOM locations from using the same HOTOSM point
- Retain matching candidates and decisions as audit data
- Export validated spatial locations as a derived GeoPackage
- Display five population indicators using a common proportional-symbol scale
- Create an interactive atlas with information, legend and layer-control panels

- IOMおよびHOTOSMの元データを読み込み、検証する
- 行政区分および人口指標の列を標準化する
- 人口値および元データの合計値を検証する
- 共通するADM3内で、IOM地点とHOTOSM集落ポイントを照合する
- 複数のIOM地点が同一のHOTOSMポイントを使用することを防ぐ
- 照合候補と採用判定を監査用データとして保存する
- 検証済み地点を派生GeoPackageとして保存する
- 共通の比例円基準を用いて5つの人口指標を表示する
- 説明、凡例およびレイヤーコントロールを備えたAtlasを作成する

Workflow

#### English

1. Define and validate the project paths
2. Read the IOM baseline and HOTOSM populated-place datasets
3. Validate source columns, CRS and geometries
4. Standardise IOM column names
5. Convert and validate population indicators
6. Prepare location records for matching
7. Normalise IOM and HOTOSM settlement names
8. Generate candidate pairs within common ADM3 areas
9. Calculate text-similarity scores
10. Assign unique HOTOSM points to accepted IOM locations
11. Validate matched and excluded records
12. Save the matching audit table and validated GeoPackage
13. Prepare five population-indicator layers
14. Create and export the interactive atlas

#### 日本語

1. プロジェクトのパスを定義し、検証する
2. IOMベースラインとHOTOSM集落データを読み込む
3. 元データの列、CRSおよびジオメトリを検証する
4. IOMの列名を標準化する
5. 人口指標を数値へ変換し、検証する
6. 照合対象となる地点データを準備する
7. IOMおよびHOTOSMの集落名を正規化する
8. 共通するADM3内で候補ペアを作成する
9. 文字列類似度を計算する
10. 採用されたIOM地点へ一意のHOTOSMポイントを割り当てる
11. 照合地点と除外地点を検証する
12. 照合監査表および検証済みGeoPackageを保存する
13. 5つの人口指標レイヤーを準備する
14. インタラクティブAtlasを作成し、保存する

Data

Displacement and return data:
- iom_displacement_return_2026.xlsx
- Source: International Organization for Migration (IOM)

Populated-place data:
- hotosm_populated_places.gpkg
- Source: Humanitarian OpenStreetMap Team (HOTOSM) / OpenStreetMap contributors

Administrative boundary data:
- syr_admin0.geojson
- syr_admin1.geojson
- Source: HDX OCHA, Syria administrative boundaries

Hydrography data:
- syr_rivers_3857.geojson
- syr_lakes_4326.geojson
- Source: OpenStreetMap contributors

Data Scope and Limitations

- IOM locations are mapped using matched HOTOSM populated-place coordinates
- The mapped coordinates are reference locations derived from the matching process and are not original IOM coordinates
- Name similarity does not by itself prove that two records describe the same settlement
- Matching is restricted to locations sharing the same ADM3 Pcode
- Only records satisfying the acceptance and uniqueness criteria are used in the primary population layers
- Review, low-confidence, unmatched and conflicting records remain available in the audit output
- Population indicators represent the definitions and reporting period of the source IOM dataset
- The atlas does not represent real-time displacement or return conditions

- IOM地点は、照合されたHOTOSM集落ポイントの座標を用いて表示します
- 地図上の座標は照合処理から得られた参照地点であり、IOMが提供した元座標ではありません
- 地名の類似だけでは、2つのレコードが同一集落を示すことを証明できません
- 照合は同一のADM3 Pcodeを持つ地点間に限定します
- 採用基準および一意性基準を満たす地点のみを主要人口レイヤーに使用します
- 要確認、低信頼、未照合および競合するレコードは、監査用出力に残します
- 人口指標はIOM元データの定義および報告期間に基づきます
- このAtlasはリアルタイムの避難・帰還状況を示すものではありません

Technologies

- Python
- Pandas
- GeoPandas
- Folium
- Difflib
- Branca

In [ ]:
# 1
# Import the required libraries
# 必要なライブラリを読み込む

# Standard library
# 標準ライブラリ

import math
import re

from difflib import SequenceMatcher
from html import escape
from pathlib import Path


# Tabular and spatial data processing
# 表形式データおよび空間データの処理

import geopandas as gpd
import pandas as pd


# Interactive web mapping
# インタラクティブWeb地図の作成

import folium

from branca.element import Element
from folium.plugins import Fullscreen, MiniMap

In [ ]:
# 2
# Define and validate the project input and output paths
# プロジェクトの入力・出力パスを定義し、検証する

PROJECT_ROOT = Path.cwd().parents[1]

DATA_DIR = PROJECT_ROOT / "02_DATA"

PROJECT_DIR = PROJECT_ROOT / "01_PROJECTS" / "06_DISPLACEMENT_RETURN_ATLAS"

OUTPUT_DIR = PROJECT_DIR / "outputs"

iom_path = DATA_DIR / "TABULAR" / "iom_displacement_return_2026.xlsx"

hotosm_path = DATA_DIR / "VECTOR" / "hotosm_populated_places.gpkg"

admin0_path = DATA_DIR / "VECTOR" / "syr_admin0.geojson"

admin1_path = DATA_DIR / "VECTOR" / "syr_admin1.geojson"

rivers_path = DATA_DIR / "VECTOR" / "syr_rivers_3857.geojson"

water_bodies_path = DATA_DIR / "VECTOR" / "syr_lakes_4326.geojson"

matching_audit_path = OUTPUT_DIR / "iom_hotosm_location_matching_audit.csv"

validated_locations_path = (
    OUTPUT_DIR / "iom_displacement_return_locations_validated.gpkg"
)

output_path = PROJECT_DIR / "02_syria_displacement_and_return_atlas_jp.html"

required_input_paths = {
    "IOM displacement and return data": iom_path,
    "HOTOSM populated places": hotosm_path,
    "Country boundary": admin0_path,
    "Governorate boundaries": admin1_path,
    "River features": rivers_path,
    "Mapped water bodies": water_bodies_path,
}

for dataset_name, dataset_path in required_input_paths.items():

    if not dataset_path.exists():

        raise FileNotFoundError(f"{dataset_name} was not found: " f"{dataset_path}")

OUTPUT_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

print("Input path validation: passed")

for dataset_name, dataset_path in required_input_paths.items():

    print(f"{dataset_name}: {dataset_path}")

print(f"Derived output directory: {OUTPUT_DIR}")

In [ ]:
# 3
# Read the IOM and HOTOSM source datasets
# IOMおよびHOTOSMの元データを読み込む

IOM_SHEET_NAME = "Short Baseline"
HOTOSM_LAYER_NAME = "populated_places"

iom = pd.read_excel(
    iom_path,
    sheet_name=IOM_SHEET_NAME,
)

hot = gpd.read_file(
    hotosm_path,
    layer=HOTOSM_LAYER_NAME,
)

if iom.empty:

    raise ValueError("The IOM source dataset contains no records.")

if hot.empty:

    raise ValueError("The HOTOSM populated-place dataset " "contains no features.")

print(f"IOM source rows: {len(iom):,}")

print(f"IOM source columns: {len(iom.columns):,}")

print(f"HOTOSM source features: {len(hot):,}")

print(f"HOTOSM source columns: {len(hot.columns):,}")

print(f"HOTOSM source CRS: {hot.crs}")

print("\nHOTOSM geometry types:")

print(hot.geom_type.value_counts(dropna=False))

In [ ]:
# 4
# Validate the source structures, CRS and geometries
# 元データの構造、CRSおよびジオメトリを検証する

if iom.columns.duplicated().any():

    duplicated_iom_columns = iom.columns[iom.columns.duplicated(keep=False)].tolist()

    raise ValueError(
        "The IOM dataset contains duplicated columns: " f"{duplicated_iom_columns}"
    )

required_hot_columns = {
    "name_en",
    "place",
    "adm3_pcode",
    "geometry",
}

missing_hot_columns = required_hot_columns - set(hot.columns)

if missing_hot_columns:

    raise ValueError(
        "The HOTOSM dataset is missing columns: " f"{sorted(missing_hot_columns)}"
    )

if hot.crs is None:

    raise ValueError("The HOTOSM populated-place dataset " "has no defined CRS.")

if hot.crs.to_epsg() != 4326:

    raise ValueError(
        "The HOTOSM populated-place dataset "
        f"was expected to use EPSG:4326, "
        f"but its CRS is {hot.crs}."
    )

if hot.geometry.isna().any():

    raise ValueError("The HOTOSM dataset contains " "missing geometries.")

if hot.geometry.is_empty.any():

    raise ValueError("The HOTOSM dataset contains " "empty geometries.")

if not hot.geometry.is_valid.all():

    invalid_hot_geometry_count = int((~hot.geometry.is_valid).sum())

    raise ValueError(
        "The HOTOSM dataset contains "
        f"{invalid_hot_geometry_count:,} "
        "invalid geometries."
    )

# Inspect the geometry types contained in the HOTOSM dataset.
# HOTOSMデータに含まれるジオメトリ形式を確認する

hotosm_geometry_summary = hot.geom_type.value_counts(dropna=False).sort_index()

print("HOTOSM source geometry composition:")

print(hotosm_geometry_summary)


# Retain only point features for settlement matching.
# 集落照合に使用するPoint形式の地物のみを抽出する

source_hotosm_feature_count = len(hot)

point_geometry_mask = hot.geom_type == "Point"

excluded_non_point_count = int((~point_geometry_mask).sum())

hot = hot.loc[point_geometry_mask].copy().reset_index(drop=True)

if hot.empty:
    raise ValueError("No point features remain in the HOTOSM dataset.")

if not hot.geom_type.eq("Point").all():
    raise ValueError(
        "The filtered HOTOSM dataset still contains " "non-point geometries."
    )

print("HOTOSM source features: " f"{source_hotosm_feature_count:,}")

print("HOTOSM point features retained: " f"{len(hot):,}")

print("HOTOSM non-point features excluded: " f"{excluded_non_point_count:,}")

print("Source structure validation: passed")

print(f"HOTOSM Point features: {len(hot):,}")

print(f"HOTOSM CRS: {hot.crs}")

print(f"Missing HOTOSM geometries: " f"{int(hot.geometry.isna().sum()):,}")

print(f"Invalid HOTOSM geometries: " f"{int((~hot.geometry.is_valid).sum()):,}")

In [ ]:
# 5
# Standardise and validate the IOM column names
# IOMの列名を標準化し、検証する

iom.columns = (
    iom.columns.astype("string")
    .str.strip()
    .str.replace(
        r"\s+",
        " ",
        regex=True,
    )
)

iom_column_map = {
    "Governorate Pcode": "adm1_pcode",
    "Governorate": "adm1_name",
    "District Pcode": "adm2_pcode",
    "District": "adm2_name",
    "Sub-District Pcode": "adm3_pcode",
    "Sub-district": "adm3_name",
    "admin 4 Pcode": "adm4_pcode",
    "admin4": "adm4_name",
    "location Type": "location_type",
    "Location Name": "location_name",
    "Location Pcode": "location_pcode",
    "Residents": "residents",
    ("IDP Returnees from December 2024 " "until now"): "idp_returnees_dec2024",
    ("IDP Returnees only in 2026"): "idp_returnees_2026",
    "IDPs": "idps",
    (
        "Arrivals from abroad to place of " "originfrom December 2024 until now"
    ): "arrivals_origin_dec2024",
    ("Arrivals from abroad to place of " "origin only in 2026"): "arrivals_origin_2026",
    (
        "Arrivals from abroad to another place " "from December 2024 until now"
    ): "arrivals_other_dec2024",
    ("Arrivals from abroad to another place " "only in 2026"): "arrivals_other_2026",
    "Total Population": "total_population",
}

missing_source_iom_columns = set(iom_column_map) - set(iom.columns)

if missing_source_iom_columns:

    raise ValueError(
        "The IOM source dataset is missing columns: "
        f"{sorted(missing_source_iom_columns)}"
    )

iom = iom.rename(columns=iom_column_map)

required_iom_columns = [
    "adm1_pcode",
    "adm1_name",
    "adm2_pcode",
    "adm2_name",
    "adm3_pcode",
    "adm3_name",
    "adm4_pcode",
    "adm4_name",
    "location_type",
    "location_name",
    "location_pcode",
    "residents",
    "idp_returnees_dec2024",
    "idp_returnees_2026",
    "idps",
    "arrivals_origin_dec2024",
    "arrivals_origin_2026",
    "arrivals_other_dec2024",
    "arrivals_other_2026",
    "total_population",
]

missing_standardised_columns = set(required_iom_columns) - set(iom.columns)

if missing_standardised_columns:

    raise ValueError(
        "The standardised IOM dataset "
        "is missing columns: "
        f"{sorted(missing_standardised_columns)}"
    )

print("IOM column standardisation: passed")

print(f"Standardised IOM columns: " f"{len(required_iom_columns):,}")

for column_name in required_iom_columns:

    print(column_name)

In [ ]:
# 6
# Convert and validate the IOM population indicators
# IOMの人口指標を数値へ変換し、検証する

population_columns = [
    "residents",
    "idp_returnees_dec2024",
    "idp_returnees_2026",
    "idps",
    "arrivals_origin_dec2024",
    "arrivals_origin_2026",
    "arrivals_other_dec2024",
    "arrivals_other_2026",
    "total_population",
]

for column_name in population_columns:

    iom[column_name] = pd.to_numeric(
        iom[column_name],
        errors="coerce",
    )


# Validate missing and non-numeric values.
# 欠損値および数値変換できなかった値を検証する

missing_population_values = iom[population_columns].isna().sum()

invalid_missing_summary = missing_population_values[missing_population_values > 0]

if not invalid_missing_summary.empty:

    raise ValueError(
        "One or more population columns contain "
        "missing or non-numeric values:\n" + invalid_missing_summary.to_string()
    )


# Validate finite population values.
# 無限値を含まないことを検証する

non_finite_population_values = {}

for column_name in population_columns:

    finite_mask = iom[column_name].apply(math.isfinite)

    non_finite_count = int((~finite_mask).sum())

    if non_finite_count > 0:

        non_finite_population_values[column_name] = non_finite_count

if non_finite_population_values:

    raise ValueError(
        "Non-finite population values were found: " + str(non_finite_population_values)
    )


# Validate non-negative population values.
# 人口値が負数でないことを検証する

negative_population_values = iom[population_columns].lt(0).sum()

invalid_negative_summary = negative_population_values[negative_population_values > 0]

if not invalid_negative_summary.empty:

    raise ValueError(
        "Negative population values were found:\n"
        + invalid_negative_summary.to_string()
    )


# Validate the 2026 indicator subsets.
# 2026年の値が対応する累計値を超えていないことを検証する

invalid_returnee_subset = iom["idp_returnees_2026"] > iom["idp_returnees_dec2024"]

if invalid_returnee_subset.any():

    raise ValueError(
        "IDP returnees recorded for 2026 exceed "
        "the corresponding totals since December 2024."
    )

invalid_origin_arrival_subset = (
    iom["arrivals_origin_2026"] > iom["arrivals_origin_dec2024"]
)

if invalid_origin_arrival_subset.any():

    raise ValueError(
        "2026 origin-arrival values exceed "
        "the corresponding totals since December 2024."
    )

invalid_other_arrival_subset = (
    iom["arrivals_other_2026"] > iom["arrivals_other_dec2024"]
)

if invalid_other_arrival_subset.any():

    raise ValueError(
        "2026 other-place arrival values exceed "
        "the corresponding totals since December 2024."
    )


# Recalculate and verify the reported total population.
# 構成項目から総人口を再計算し、報告値と照合する

calculated_total_population = (
    iom["residents"]
    + iom["idp_returnees_dec2024"]
    + iom["idps"]
    + iom["arrivals_origin_dec2024"]
    + iom["arrivals_other_dec2024"]
)

total_population_match = calculated_total_population.eq(iom["total_population"])

total_population_mismatch_count = int((~total_population_match).sum())

if total_population_mismatch_count > 0:

    raise ValueError(
        "The calculated and reported total population "
        "values differ for "
        f"{total_population_mismatch_count:,} records."
    )


# Prepare the validation summary.
# 検証結果の要約を準備する

negative_population_count = int(iom[population_columns].lt(0).sum().sum())

print("Population-indicator validation: passed")

print(f"Validated IOM rows: {len(iom):,}")

print("Negative population values: " f"{negative_population_count:,}")

print("Total-population mismatches: " f"{total_population_mismatch_count:,}")

print("\nPopulation indicator summary:")

print(
    iom[population_columns]
    .agg(
        [
            "min",
            "max",
            "sum",
        ]
    )
    .T
)

In [ ]:
# 7
# Prepare the IOM and HOTOSM datasets for location matching
# IOMとHOTOSMの地点照合用データを準備する

iom_match = (
    iom.copy()
    .reset_index(drop=False)
    .rename(
        columns={
            "index": "iom_source_index",
        }
    )
)

hot_match = (
    hot.copy()
    .reset_index(drop=False)
    .rename(
        columns={
            "index": "hot_source_index",
        }
    )
)


# Standardise the administrative identifiers.
# 行政区域コードの文字形式を統一する

iom_match["adm3_pcode"] = iom_match["adm3_pcode"].astype("string").str.strip()

hot_match["adm3_pcode"] = hot_match["adm3_pcode"].astype("string").str.strip()


# Standardise the IOM location identifiers and names.
# IOMの地点コードと地点名を整える

iom_match["location_pcode"] = iom_match["location_pcode"].astype("string").str.strip()

iom_match["location_name"] = iom_match["location_name"].astype("string").str.strip()


# Select the preferred HOTOSM settlement name.
# HOTOSMから照合に使用する集落名を選択する

hot_name_candidates = hot_match[
    [
        "name_en",
        "name_latin",
        "name",
    ]
].copy()

for column_name in hot_name_candidates.columns:

    hot_name_candidates[column_name] = (
        hot_name_candidates[column_name]
        .astype("string")
        .str.strip()
        .replace(
            "",
            pd.NA,
        )
    )

hot_match["matching_name"] = (
    hot_name_candidates["name_en"]
    .fillna(hot_name_candidates["name_latin"])
    .fillna(hot_name_candidates["name"])
)


# Validate unique source identifiers.
# 元データの識別子が一意であることを検証する

if iom_match["location_pcode"].duplicated().any():

    raise ValueError("Duplicate IOM Location Pcodes were found.")

if hot_match["id"].duplicated().any():

    raise ValueError("Duplicate HOTOSM feature IDs were found.")


# Remove HOTOSM points that cannot participate in matching.
# 照合に使用できないHOTOSM地点を除外する

missing_hot_adm3_mask = hot_match["adm3_pcode"].isna() | hot_match["adm3_pcode"].eq("")

missing_hot_name_mask = hot_match["matching_name"].isna() | hot_match[
    "matching_name"
].eq("")

excluded_hot_adm3_count = int(missing_hot_adm3_mask.sum())

excluded_hot_name_count = int(missing_hot_name_mask.sum())

hot_match = (
    hot_match.loc[~missing_hot_adm3_mask & ~missing_hot_name_mask]
    .copy()
    .reset_index(drop=True)
)

if hot_match.empty:

    raise ValueError("No HOTOSM point features remain for matching.")


# Identify ADM3 areas represented by both datasets.
# 両方のデータに存在するADM3区域を確認する

common_adm3_pcodes = set(iom_match["adm3_pcode"].dropna()) & set(
    hot_match["adm3_pcode"].dropna()
)

iom_rows_in_common_adm3 = int(iom_match["adm3_pcode"].isin(common_adm3_pcodes).sum())

print(f"IOM matching records: {len(iom_match):,}")

print(f"HOTOSM matching points: {len(hot_match):,}")

print("HOTOSM points excluded for missing ADM3: " f"{excluded_hot_adm3_count:,}")

print("HOTOSM points excluded for missing names: " f"{excluded_hot_name_count:,}")

print(f"Common ADM3 areas: {len(common_adm3_pcodes):,}")

print("IOM records inside common ADM3 areas: " f"{iom_rows_in_common_adm3:,}")

In [ ]:
# 8
# Define and apply the location-name normalisation function
# 地点名の正規化関数を定義し、適用する


def normalise_location_name(value):

    if pd.isna(value):
        return ""

    normalised_value = str(value).casefold().strip()

    normalised_value = re.sub(
        r"[_\W]+",
        " ",
        normalised_value,
        flags=re.UNICODE,
    )

    normalised_value = re.sub(
        r"\s+",
        " ",
        normalised_value,
    ).strip()

    return normalised_value


iom_match["normalised_location_name"] = iom_match["location_name"].apply(
    normalise_location_name
)

hot_match["normalised_matching_name"] = hot_match["matching_name"].apply(
    normalise_location_name
)


# Validate the normalised matching names.
# 正規化後の地点名を検証する

blank_iom_name_count = int(iom_match["normalised_location_name"].eq("").sum())

blank_hot_name_count = int(hot_match["normalised_matching_name"].eq("").sum())

if blank_iom_name_count > 0:

    raise ValueError(
        "Normalisation produced blank names for "
        f"{blank_iom_name_count:,} IOM records."
    )

if blank_hot_name_count > 0:

    raise ValueError(
        "Normalisation produced blank names for "
        f"{blank_hot_name_count:,} HOTOSM records."
    )

print("Location-name normalisation: passed")

print(
    "Unique normalised IOM names: "
    f"{iom_match['normalised_location_name'].nunique():,}"
)

print(
    "Unique normalised HOTOSM names: "
    f"{hot_match['normalised_matching_name'].nunique():,}"
)

print(
    iom_match[
        [
            "location_name",
            "normalised_location_name",
        ]
    ].head()
)

In [ ]:
# 9
# Define the settlement-name similarity function
# 集落名の類似度を計算する関数を定義する


def calculate_name_similarity(
    left_name,
    right_name,
):

    if not left_name or not right_name:
        return 0.0

    if left_name == right_name:
        return 100.0

    direct_score = (
        SequenceMatcher(
            None,
            left_name,
            right_name,
        ).ratio()
        * 100
    )

    left_tokens = " ".join(sorted(left_name.split()))

    right_tokens = " ".join(sorted(right_name.split()))

    token_order_score = (
        SequenceMatcher(
            None,
            left_tokens,
            right_tokens,
        ).ratio()
        * 100
    )

    return round(
        max(
            direct_score,
            token_order_score,
        ),
        2,
    )


# Verify the similarity function using controlled examples.
# 既知の例を使って類似度関数を検証する

similarity_tests = pd.DataFrame(
    [
        {
            "left_name": "al mayadin",
            "right_name": "al mayadin",
        },
        {
            "left_name": "abu kamal",
            "right_name": "al bukamal",
        },
        {
            "left_name": "deir ez zor",
            "right_name": "deir ez zor",
        },
    ]
)

similarity_tests["similarity_score"] = similarity_tests.apply(
    lambda row: calculate_name_similarity(
        row["left_name"],
        row["right_name"],
    ),
    axis=1,
)

if (
    similarity_tests.loc[
        0,
        "similarity_score",
    ]
    != 100.0
):

    raise ValueError("The similarity function failed the exact-match test.")

print("Name-similarity function: passed")

print(similarity_tests)

In [ ]:
# 10
# Generate the three highest-scoring HOTOSM candidates within each ADM3
# 各IOM地点について同一ADM3内のHOTOSM上位3候補を作成する

TOP_CANDIDATES_PER_IOM = 3

hot_groups_by_adm3 = {
    adm3_pcode: group.copy()
    for adm3_pcode, group in hot_match.groupby(
        "adm3_pcode",
        sort=False,
    )
}

candidate_rows = []

for iom_row in iom_match.itertuples(index=False):

    hot_adm3_group = hot_groups_by_adm3.get(iom_row.adm3_pcode)

    if hot_adm3_group is None:
        continue

    scored_candidates = []

    for hot_row in hot_adm3_group.itertuples(index=False):

        similarity_score = calculate_name_similarity(
            iom_row.normalised_location_name,
            hot_row.normalised_matching_name,
        )

        scored_candidates.append(
            {
                "iom_source_index": iom_row.iom_source_index,
                "location_pcode": iom_row.location_pcode,
                "iom_location_name": iom_row.location_name,
                "adm3_pcode": iom_row.adm3_pcode,
                "hot_source_index": hot_row.hot_source_index,
                "hot_id": hot_row.id,
                "hot_matching_name": hot_row.matching_name,
                "iom_normalised_name": (iom_row.normalised_location_name),
                "hot_normalised_name": (hot_row.normalised_matching_name),
                "similarity_score": similarity_score,
            }
        )

    scored_candidates = sorted(
        scored_candidates,
        key=lambda candidate: (
            -candidate["similarity_score"],
            str(candidate["hot_id"]),
        ),
    )

    for candidate_rank, candidate in enumerate(
        scored_candidates[:TOP_CANDIDATES_PER_IOM],
        start=1,
    ):

        candidate["candidate_rank"] = candidate_rank

        candidate_rows.append(candidate)

candidate_matches = pd.DataFrame(candidate_rows)

if candidate_matches.empty:

    raise ValueError("No location-matching candidates were generated.")

expected_candidate_columns = {
    "iom_source_index",
    "location_pcode",
    "iom_location_name",
    "adm3_pcode",
    "hot_source_index",
    "hot_id",
    "hot_matching_name",
    "similarity_score",
    "candidate_rank",
}

missing_candidate_columns = expected_candidate_columns - set(candidate_matches.columns)

if missing_candidate_columns:

    raise ValueError(
        "The candidate table is missing required columns: "
        f"{sorted(missing_candidate_columns)}"
    )

print(f"Candidate rows generated: {len(candidate_matches):,}")

print(
    "IOM records with candidates: "
    f"{candidate_matches['iom_source_index'].nunique():,}"
)

print(
    candidate_matches[
        [
            "location_pcode",
            "iom_location_name",
            "hot_matching_name",
            "similarity_score",
            "candidate_rank",
        ]
    ].head(10)
)

In [ ]:
# 11
# Classify the location-matching candidates by confidence
# 地点照合候補を信頼度別に分類する

HIGH_CONFIDENCE_THRESHOLD = 85.0
REVIEW_THRESHOLD = 70.0

candidate_matches["name_exact_match"] = (
    candidate_matches["iom_normalised_name"] == candidate_matches["hot_normalised_name"]
)

candidate_matches["match_status"] = "Low confidence"

candidate_matches.loc[
    candidate_matches["similarity_score"].ge(REVIEW_THRESHOLD),
    "match_status",
] = "Review"

candidate_matches.loc[
    candidate_matches["similarity_score"].ge(HIGH_CONFIDENCE_THRESHOLD),
    "match_status",
] = "High confidence"

candidate_matches.loc[
    candidate_matches["name_exact_match"],
    "match_status",
] = "Exact"

candidate_matches["best_candidate"] = candidate_matches["candidate_rank"] == 1

candidate_matches = candidate_matches.sort_values(
    [
        "iom_source_index",
        "candidate_rank",
    ]
).reset_index(drop=True)

print("Candidate confidence classification:")

print(candidate_matches["match_status"].value_counts())

print("\nBest-candidate classification:")

print(
    candidate_matches.loc[
        candidate_matches["best_candidate"],
        "match_status",
    ].value_counts()
)

In [ ]:
# 12
# Validate and summarise the candidate-generation results
# 候補作成結果を検証し、要約する

best_candidates = (
    candidate_matches.loc[candidate_matches["best_candidate"]]
    .copy()
    .reset_index(drop=True)
)

iom_indices_with_candidates = set(best_candidates["iom_source_index"])

iom_without_candidates = (
    iom_match.loc[
        ~iom_match["iom_source_index"].isin(iom_indices_with_candidates),
        [
            "iom_source_index",
            "location_pcode",
            "location_name",
            "adm1_name",
            "adm2_name",
            "adm3_name",
            "adm3_pcode",
        ],
    ]
    .copy()
    .reset_index(drop=True)
)

iom_candidate_count = candidate_matches.groupby("iom_source_index").size()

if (iom_candidate_count > TOP_CANDIDATES_PER_IOM).any():

    raise ValueError("One or more IOM records exceed the defined " "candidate limit.")

candidate_coverage_count = len(best_candidates)

candidate_coverage_percentage = candidate_coverage_count / len(iom_match) * 100

exact_best_candidate_count = int(best_candidates["name_exact_match"].sum())

print("Candidate-generation validation: passed")

print("IOM records evaluated: " f"{len(iom_match):,}")

print("IOM records with candidates: " f"{candidate_coverage_count:,}")

print("IOM records without candidates: " f"{len(iom_without_candidates):,}")

print("Candidate coverage: " f"{candidate_coverage_percentage:.2f}%")

print("Exact best candidates: " f"{exact_best_candidate_count:,}")

print("\nBest-candidate score summary:")

print(best_candidates["similarity_score"].describe())

if not iom_without_candidates.empty:

    print("\nIOM records without HOTOSM candidates:")

    print(iom_without_candidates)

In [ ]:
# 13
# Assign eligible candidates using deterministic one-to-one matching
# 採用可能な候補を使用して決定的な一対一照合を行う

eligible_candidates = candidate_matches.loc[
    candidate_matches["name_exact_match"]
    | candidate_matches["similarity_score"].ge(HIGH_CONFIDENCE_THRESHOLD)
].copy()

eligible_candidates["confidence_priority"] = (
    eligible_candidates["match_status"]
    .map(
        {
            "Exact": 0,
            "High confidence": 1,
        }
    )
    .fillna(2)
    .astype(int)
)

eligible_candidates = eligible_candidates.sort_values(
    [
        "confidence_priority",
        "similarity_score",
        "candidate_rank",
        "location_pcode",
        "hot_id",
    ],
    ascending=[
        True,
        False,
        True,
        True,
        True,
    ],
).reset_index(drop=True)


# Track identifiers that have already been assigned.
# すでに割り当てられた識別子を記録する

assigned_iom_indices = set()
assigned_hot_ids = set()
accepted_assignment_rows = []

for candidate in eligible_candidates.to_dict(orient="records"):

    iom_source_index = candidate["iom_source_index"]

    hot_id = candidate["hot_id"]

    if iom_source_index in assigned_iom_indices:
        continue

    if hot_id in assigned_hot_ids:
        continue

    candidate["assignment_order"] = len(accepted_assignment_rows) + 1

    accepted_assignment_rows.append(candidate)

    assigned_iom_indices.add(iom_source_index)

    assigned_hot_ids.add(hot_id)

accepted_assignments = pd.DataFrame(accepted_assignment_rows)

if accepted_assignments.empty:

    raise ValueError("No eligible one-to-one assignments were accepted.")

accepted_assignments = (
    accepted_assignments.drop(
        columns=[
            "confidence_priority",
        ],
        errors="ignore",
    )
    .sort_values("iom_source_index")
    .reset_index(drop=True)
)

print("One-to-one candidate assignment: completed")

print("Eligible candidate rows: " f"{len(eligible_candidates):,}")

print("Accepted one-to-one assignments: " f"{len(accepted_assignments):,}")

print(
    "Unique assigned IOM records: "
    f"{accepted_assignments['iom_source_index'].nunique():,}"
)

print("Unique assigned HOTOSM points: " f"{accepted_assignments['hot_id'].nunique():,}")

In [ ]:
# 14
# Create a decision table for all IOM location records
# すべてのIOM地点について照合判断表を作成する

best_candidate_summary = best_candidates[
    [
        "iom_source_index",
        "hot_id",
        "hot_matching_name",
        "similarity_score",
        "match_status",
    ]
].rename(
    columns={
        "hot_id": "best_candidate_hot_id",
        "hot_matching_name": ("best_candidate_hot_name"),
        "similarity_score": ("best_candidate_score"),
        "match_status": ("best_candidate_status"),
    }
)

accepted_assignment_summary = accepted_assignments[
    [
        "iom_source_index",
        "hot_id",
        "hot_matching_name",
        "similarity_score",
        "match_status",
        "candidate_rank",
    ]
].rename(
    columns={
        "hot_id": "accepted_hot_id",
        "hot_matching_name": ("accepted_hot_name"),
        "similarity_score": ("accepted_similarity_score"),
        "match_status": ("accepted_match_status"),
        "candidate_rank": ("accepted_candidate_rank"),
    }
)

matching_decisions = (
    iom_match[
        [
            "iom_source_index",
            "location_pcode",
            "location_name",
            "adm1_name",
            "adm2_name",
            "adm3_name",
            "adm3_pcode",
        ]
    ]
    .merge(
        best_candidate_summary,
        on="iom_source_index",
        how="left",
        validate="one_to_one",
    )
    .merge(
        accepted_assignment_summary,
        on="iom_source_index",
        how="left",
        validate="one_to_one",
    )
)

matching_decisions["matching_decision"] = "Review required"

no_candidate_mask = matching_decisions["best_candidate_hot_id"].isna()

accepted_match_mask = matching_decisions["accepted_hot_id"].notna()

eligible_but_unassigned_mask = (
    matching_decisions["best_candidate_score"].ge(HIGH_CONFIDENCE_THRESHOLD)
    & matching_decisions["accepted_hot_id"].isna()
)

matching_decisions.loc[
    no_candidate_mask,
    "matching_decision",
] = "No candidate"

matching_decisions.loc[
    eligible_but_unassigned_mask,
    "matching_decision",
] = "Unresolved duplicate conflict"

matching_decisions.loc[
    accepted_match_mask,
    "matching_decision",
] = "Accepted"

print("Matching-decision table: completed")

print(matching_decisions["matching_decision"].value_counts())

print("\nAccepted match composition:")

print(
    matching_decisions.loc[
        matching_decisions["matching_decision"].eq("Accepted"),
        "accepted_match_status",
    ].value_counts()
)

In [ ]:
# 15
# Validate accepted assignments and unresolved IOM records
# 採用された照合と未解決のIOM地点を検証する

duplicate_iom_assignment_count = int(
    accepted_assignments["iom_source_index"].duplicated().sum()
)

duplicate_hot_assignment_count = int(accepted_assignments["hot_id"].duplicated().sum())

below_threshold_assignment_count = int(
    (
        ~accepted_assignments["name_exact_match"]
        & accepted_assignments["similarity_score"].lt(HIGH_CONFIDENCE_THRESHOLD)
    ).sum()
)

if duplicate_iom_assignment_count > 0:

    raise ValueError(
        "One or more IOM records received multiple " "accepted assignments."
    )

if duplicate_hot_assignment_count > 0:

    raise ValueError(
        "One or more HOTOSM points were assigned " "to multiple IOM records."
    )

if below_threshold_assignment_count > 0:

    raise ValueError(
        "One or more accepted assignments fall below "
        "the automatic acceptance threshold."
    )

unresolved_iom_records = (
    matching_decisions.loc[~matching_decisions["matching_decision"].eq("Accepted")]
    .copy()
    .reset_index(drop=True)
)

accepted_assignment_count = len(accepted_assignments)

unresolved_iom_count = len(unresolved_iom_records)

automatic_acceptance_percentage = accepted_assignment_count / len(iom_match) * 100

print("One-to-one assignment validation: passed")

print("Duplicate IOM assignments: " f"{duplicate_iom_assignment_count:,}")

print("Duplicate HOTOSM assignments: " f"{duplicate_hot_assignment_count:,}")

print("Below-threshold accepted assignments: " f"{below_threshold_assignment_count:,}")

print("Automatically accepted IOM records: " f"{accepted_assignment_count:,}")

print("Unresolved IOM records: " f"{unresolved_iom_count:,}")

print("Automatic acceptance coverage: " f"{automatic_acceptance_percentage:.2f}%")

print("\nUnresolved-decision composition:")

print(unresolved_iom_records["matching_decision"].value_counts())

In [ ]:
# 16
# Attach HOTOSM geometry and settlement attributes to accepted matches
# 採用された照合へHOTOSMの座標と集落属性を付与する

hot_geometry_lookup = hot.reset_index(drop=False).rename(
    columns={
        "index": "hot_source_index",
        "name": "hot_name",
        "name_en": "hot_name_en",
        "name_latin": "hot_name_latin",
        "place": "hot_place",
        "adm1_pcode": "hot_adm1_pcode",
        "adm1_name": "hot_adm1_name",
        "adm2_pcode": "hot_adm2_pcode",
        "adm2_name": "hot_adm2_name",
        "adm3_pcode": "hot_adm3_pcode",
        "adm3_name": "hot_adm3_name",
    }
)

hot_geometry_columns = [
    "hot_source_index",
    "hot_name",
    "hot_name_en",
    "hot_name_latin",
    "hot_place",
    "hot_adm1_pcode",
    "hot_adm1_name",
    "hot_adm2_pcode",
    "hot_adm2_name",
    "hot_adm3_pcode",
    "hot_adm3_name",
    "geometry",
]

matched_locations = accepted_assignments.merge(
    hot_geometry_lookup[hot_geometry_columns],
    on="hot_source_index",
    how="left",
    validate="one_to_one",
)

matched_locations = gpd.GeoDataFrame(
    matched_locations,
    geometry="geometry",
    crs=hot.crs,
)

if matched_locations["geometry"].isna().any():

    missing_geometry_count = int(matched_locations["geometry"].isna().sum())

    raise ValueError(
        "HOTOSM geometry was not attached to "
        f"{missing_geometry_count:,} accepted matches."
    )

print("HOTOSM geometry attachment: passed")

print("Matched locations with geometry: " f"{len(matched_locations):,}")

print(f"Matched-location CRS: {matched_locations.crs}")

In [ ]:
# 17
# Attach the IOM administrative and population attributes
# IOMの行政属性と人口指標を照合済み地点へ付与する

iom_attribute_lookup = iom.reset_index(drop=False).rename(
    columns={
        "index": "iom_source_index",
    }
)

candidate_only_columns = [
    "location_pcode",
    "iom_location_name",
    "adm3_pcode",
]

matched_locations_core = matched_locations.drop(
    columns=candidate_only_columns,
    errors="ignore",
)

validated_locations = matched_locations_core.merge(
    iom_attribute_lookup,
    on="iom_source_index",
    how="left",
    validate="one_to_one",
)

validated_locations = gpd.GeoDataFrame(
    validated_locations,
    geometry="geometry",
    crs=matched_locations.crs,
)

required_attached_columns = {
    "location_pcode",
    "location_name",
    "residents",
    "idp_returnees_dec2024",
    "idp_returnees_2026",
    "idps",
    "total_population",
    "geometry",
}

missing_attached_columns = required_attached_columns - set(validated_locations.columns)

if missing_attached_columns:

    raise ValueError(
        "The validated location dataset is missing "
        "required columns: "
        f"{sorted(missing_attached_columns)}"
    )

print("IOM attribute attachment: passed")

print("Validated matched locations: " f"{len(validated_locations):,}")

print(
    validated_locations[
        [
            "location_pcode",
            "location_name",
            "hot_matching_name",
            "similarity_score",
            "match_status",
            "residents",
            "idps",
            "idp_returnees_dec2024",
            "idp_returnees_2026",
            "total_population",
        ]
    ].head()
)

In [ ]:
# 18
# Validate the matched geospatial location dataset
# 照合済み地点GeoDataFrameを検証する

if validated_locations.empty:

    raise ValueError("The validated matched-location dataset is empty.")

if validated_locations.crs is None:

    raise ValueError("The validated matched-location dataset " "has no defined CRS.")

if validated_locations.crs.to_epsg() != 4326:

    raise ValueError("The validated matched-location dataset must " "use EPSG:4326.")

missing_validated_geometry_count = int(validated_locations["geometry"].isna().sum())

empty_validated_geometry_count = int(validated_locations["geometry"].is_empty.sum())

invalid_validated_geometry_count = int(
    (~validated_locations["geometry"].is_valid).sum()
)

non_point_validated_geometry_count = int(
    (~validated_locations.geom_type.eq("Point")).sum()
)

duplicate_location_pcode_count = int(
    validated_locations["location_pcode"].duplicated().sum()
)

duplicate_validated_hot_id_count = int(validated_locations["hot_id"].duplicated().sum())

adm3_assignment_mismatch = (
    validated_locations["adm3_pcode"]
    .astype("string")
    .ne(validated_locations["hot_adm3_pcode"].astype("string"))
)

adm3_assignment_mismatch_count = int(adm3_assignment_mismatch.sum())

if missing_validated_geometry_count > 0:

    raise ValueError("Missing geometries remain in the validated dataset.")

if empty_validated_geometry_count > 0:

    raise ValueError("Empty geometries remain in the validated dataset.")

if invalid_validated_geometry_count > 0:

    raise ValueError("Invalid geometries remain in the validated dataset.")

if non_point_validated_geometry_count > 0:

    raise ValueError("Non-point geometries remain in the validated dataset.")

if duplicate_location_pcode_count > 0:

    raise ValueError(
        "Duplicate IOM Location Pcodes remain in the " "validated dataset."
    )

if duplicate_validated_hot_id_count > 0:

    raise ValueError("Duplicate HOTOSM IDs remain in the " "validated dataset.")

if adm3_assignment_mismatch_count > 0:

    raise ValueError(
        "One or more accepted locations were assigned " "outside their IOM ADM3 area."
    )

print("Matched geospatial dataset validation: passed")

print("Validated location features: " f"{len(validated_locations):,}")

print(f"Validated CRS: {validated_locations.crs}")

print("Missing geometries: " f"{missing_validated_geometry_count:,}")

print("Invalid geometries: " f"{invalid_validated_geometry_count:,}")

print("Duplicate Location Pcodes: " f"{duplicate_location_pcode_count:,}")

print("Duplicate HOTOSM IDs: " f"{duplicate_validated_hot_id_count:,}")

print("ADM3 assignment mismatches: " f"{adm3_assignment_mismatch_count:,}")

print("\nAccepted confidence composition:")

print(validated_locations["match_status"].value_counts())

In [ ]:
# 19
# Export and verify the complete location-matching audit table
# 全IOM地点の照合監査表を保存し、検証する

matching_audit_export = (
    matching_decisions.copy()
    .sort_values(
        [
            "adm1_name",
            "adm2_name",
            "adm3_name",
            "location_name",
            "location_pcode",
        ]
    )
    .reset_index(drop=True)
)

matching_audit_export["automatic_acceptance_threshold"] = HIGH_CONFIDENCE_THRESHOLD

matching_audit_export.to_csv(
    matching_audit_path,
    index=False,
    encoding="utf-8-sig",
)

if not matching_audit_path.exists():

    raise FileNotFoundError("The location-matching audit CSV was not created.")

saved_matching_audit = pd.read_csv(
    matching_audit_path,
    encoding="utf-8-sig",
)

if len(saved_matching_audit) != len(iom_match):

    raise ValueError(
        "The saved matching audit row count does not " "match the IOM source row count."
    )

if saved_matching_audit["location_pcode"].duplicated().any():

    raise ValueError(
        "Duplicate Location Pcodes were found in the " "saved matching audit."
    )

print("Location-matching audit export: passed")

print(f"Saved audit rows: {len(saved_matching_audit):,}")

print(f"Audit CSV: {matching_audit_path}")

print("\nSaved decision composition:")

print(saved_matching_audit["matching_decision"].value_counts())

In [ ]:
# 20
# Export and verify the validated matched locations as a GeoPackage
# 検証済み照合地点をGeoPackageとして保存し、検証する

validated_locations_export = (
    validated_locations.copy()
    .sort_values(
        [
            "adm1_name",
            "adm2_name",
            "adm3_name",
            "location_name",
            "location_pcode",
        ]
    )
    .reset_index(drop=True)
)

validated_locations_export.to_file(
    validated_locations_path,
    layer="validated_locations",
    driver="GPKG",
    mode="w",
)

if not validated_locations_path.exists():

    raise FileNotFoundError("The validated location GeoPackage was not created.")

saved_validated_locations = gpd.read_file(
    validated_locations_path,
    layer="validated_locations",
)

if len(saved_validated_locations) != len(validated_locations_export):

    raise ValueError(
        "The saved GeoPackage feature count does not "
        "match the validated location count."
    )

if saved_validated_locations.crs is None:

    raise ValueError("The saved validated location GeoPackage " "has no defined CRS.")

if saved_validated_locations.crs.to_epsg() != 4326:

    raise ValueError(
        "The saved validated location GeoPackage " "does not use EPSG:4326."
    )

if saved_validated_locations["geometry"].isna().any():

    raise ValueError(
        "The saved validated location GeoPackage " "contains missing geometries."
    )

if not saved_validated_locations["geometry"].is_valid.all():

    raise ValueError(
        "The saved validated location GeoPackage " "contains invalid geometries."
    )

if saved_validated_locations["location_pcode"].duplicated().any():

    raise ValueError(
        "The saved validated location GeoPackage " "contains duplicate Location Pcodes."
    )

if saved_validated_locations["hot_id"].duplicated().any():

    raise ValueError(
        "The saved validated location GeoPackage " "contains duplicate HOTOSM IDs."
    )

print("Validated GeoPackage export: passed")

print("Saved validated features: " f"{len(saved_validated_locations):,}")

print(f"Saved CRS: {saved_validated_locations.crs}")

print(f"Validated GeoPackage: {validated_locations_path}")

In [ ]:
# 21
# Calculate the population-indicator coverage of validated locations
# 検証済み地点がカバーする人口指標の割合を計算する

atlas_indicator_columns = [
    "residents",
    "idps",
    "idp_returnees_dec2024",
    "idp_returnees_2026",
    "total_population",
]

coverage_rows = []

for indicator_name in atlas_indicator_columns:

    source_total = int(iom[indicator_name].sum())

    mapped_total = int(validated_locations[indicator_name].sum())

    if mapped_total > source_total:

        raise ValueError(
            f"The mapped {indicator_name} total exceeds "
            "the corresponding IOM source total."
        )

    if source_total > 0:

        coverage_percentage = mapped_total / source_total * 100

    else:

        coverage_percentage = 0.0

    coverage_rows.append(
        {
            "indicator": indicator_name,
            "source_total": source_total,
            "mapped_total": mapped_total,
            "coverage_percentage": coverage_percentage,
        }
    )

mapping_coverage = pd.DataFrame(coverage_rows)

if len(mapping_coverage) != len(atlas_indicator_columns):

    raise ValueError("The mapping-coverage summary is incomplete.")

print("Validated-location indicator coverage:")

print(
    mapping_coverage.to_string(
        index=False,
        formatters={
            "source_total": (lambda value: f"{value:,.0f}"),
            "mapped_total": (lambda value: f"{value:,.0f}"),
            "coverage_percentage": (lambda value: f"{value:.2f}%"),
        },
    )
)

In [ ]:
# 22
# Define a shared proportional-circle scale for all atlas layers
# 全Atlasレイヤーで共有する比例円スケールを定義する

MAX_CIRCLE_RADIUS_PIXELS = 22.0
MIN_VISIBLE_RADIUS_PIXELS = 2.0
CIRCLE_REFERENCE_PERCENTILE = 0.99

positive_scale_values = (
    validated_locations[atlas_indicator_columns]
    .where(validated_locations[atlas_indicator_columns].gt(0))
    .stack()
)

if positive_scale_values.empty:

    raise ValueError(
        "No positive population values are available "
        "for proportional-circle scaling."
    )

circle_reference_value = float(
    positive_scale_values.quantile(CIRCLE_REFERENCE_PERCENTILE)
)

if not math.isfinite(circle_reference_value) or circle_reference_value <= 0:

    raise ValueError("The proportional-circle reference value " "is invalid.")


def calculate_circle_radius(value):

    if pd.isna(value):
        return 0.0

    numeric_value = float(value)

    if numeric_value <= 0:
        return 0.0

    capped_value = min(
        numeric_value,
        circle_reference_value,
    )

    proportional_radius = MAX_CIRCLE_RADIUS_PIXELS * math.sqrt(
        capped_value / circle_reference_value
    )

    return max(
        MIN_VISIBLE_RADIUS_PIXELS,
        proportional_radius,
    )


# Validate representative circle sizes.
# 代表値を使って比例円サイズを検証する

circle_scale_test_values = [
    0,
    circle_reference_value * 0.25,
    circle_reference_value * 0.50,
    circle_reference_value,
]

circle_scale_test = pd.DataFrame({"population_value": (circle_scale_test_values)})

circle_scale_test["radius_pixels"] = circle_scale_test["population_value"].apply(
    calculate_circle_radius
)

if circle_scale_test["radius_pixels"].iloc[0] != 0.0:

    raise ValueError("A zero population value must produce " "a zero-radius circle.")

if not circle_scale_test["radius_pixels"].is_monotonic_increasing:

    raise ValueError(
        "The proportional-circle radii are not " "monotonically increasing."
    )

print("Shared proportional-circle scale: passed")

print("Circle reference percentile: " f"{CIRCLE_REFERENCE_PERCENTILE * 100:.0f}%")

print("Circle reference value: " f"{circle_reference_value:,.0f}")

print(circle_scale_test)

In [ ]:
# 23
# Prepare the five population indicator layers for the atlas
# Atlasで使用する5つの人口指標レイヤーを準備する

residents_layer = validated_locations.loc[validated_locations["residents"].gt(0)].copy()

idps_layer = validated_locations.loc[validated_locations["idps"].gt(0)].copy()

idp_returnees_dec2024_layer = validated_locations.loc[
    validated_locations["idp_returnees_dec2024"].gt(0)
].copy()

idp_returnees_2026_layer = validated_locations.loc[
    validated_locations["idp_returnees_2026"].gt(0)
].copy()

total_population_layer = validated_locations.loc[
    validated_locations["total_population"].gt(0)
].copy()

atlas_layers = {
    "Residents": {
        "data": residents_layer,
        "indicator": "residents",
        "colour": "#2A9D8F",
    },
    "IDPs": {
        "data": idps_layer,
        "indicator": "idps",
        "colour": "#E76F51",
    },
    "IDP Returnees Since December 2024": {
        "data": idp_returnees_dec2024_layer,
        "indicator": "idp_returnees_dec2024",
        "colour": "#d1908d",
    },
    "IDP Returnees in 2026": {
        "data": idp_returnees_2026_layer,
        "indicator": "idp_returnees_2026",
        "colour": "#8B1E3F",
    },
    "Total Population": {
        "data": total_population_layer,
        "indicator": "total_population",
        "colour": "#F4A261",
    },
}

for layer_name, layer_definition in atlas_layers.items():

    layer_data = layer_definition["data"]

    indicator_name = layer_definition["indicator"]

    if not isinstance(
        layer_data,
        gpd.GeoDataFrame,
    ):

        raise TypeError(f"{layer_name} was not prepared as " "a GeoDataFrame.")

    if layer_data.crs != validated_locations.crs:

        raise ValueError(f"{layer_name} does not use the validated CRS.")

    if not layer_data[indicator_name].gt(0).all():

        raise ValueError(f"{layer_name} contains zero or negative values.")

print("Atlas indicator layers: prepared")

for layer_name, layer_definition in atlas_layers.items():

    layer_data = layer_definition["data"]

    indicator_name = layer_definition["indicator"]

    print(
        f"{layer_name}: "
        f"{len(layer_data):,} locations, "
        f"{int(layer_data[indicator_name].sum()):,}"
    )

In [ ]:
# 24
# Read, validate and prepare the contextual spatial layers
# 行政界、河川および水域の背景空間レイヤーを準備する

admin0 = gpd.read_file(admin0_path)

admin1 = gpd.read_file(admin1_path)

rivers = gpd.read_file(rivers_path)

water_bodies = gpd.read_file(water_bodies_path)

context_datasets = {
    "Country boundary": admin0,
    "Governorate boundaries": admin1,
    "Rivers": rivers,
    "Mapped water bodies": water_bodies,
}

for dataset_name, dataset in context_datasets.items():

    if dataset.empty:

        raise ValueError(f"{dataset_name} contains no features.")

    if dataset.crs is None:

        raise ValueError(f"{dataset_name} has no defined CRS.")

    if dataset["geometry"].isna().any():

        raise ValueError(f"{dataset_name} contains missing geometries.")

    if dataset["geometry"].is_empty.any():

        raise ValueError(f"{dataset_name} contains empty geometries.")

    if not dataset["geometry"].is_valid.all():

        raise ValueError(f"{dataset_name} contains invalid geometries.")


# Prepare administrative display copies without date fields.
# 日付型属性を含まない行政界表示用データを作成する

admin0_web = (
    admin0[
        [
            "adm0_name",
            "adm0_pcode",
            "geometry",
        ]
    ]
    .to_crs(4326)
    .copy()
)

admin1_web = (
    admin1[
        [
            "adm1_name",
            "adm1_pcode",
            "center_lat",
            "center_lon",
            "geometry",
        ]
    ]
    .to_crs(4326)
    .copy()
)


# Simplify river features in a projected CRS.
# 投影座標系で河川表示データを簡略化する

RIVER_SIMPLIFICATION_METRES = 75.0
WATER_BODY_SIMPLIFICATION_METRES = 75.0

rivers_display_projected = rivers.to_crs(3857)[
    [
        "name",
        "name:en",
        "waterway",
        "source",
        "geometry",
    ]
].copy()

rivers_display_projected["geometry"] = rivers_display_projected["geometry"].simplify(
    RIVER_SIMPLIFICATION_METRES,
    preserve_topology=True,
)

rivers_web = rivers_display_projected.to_crs(4326)


# Simplify mapped water bodies in a projected CRS.
# 投影座標系で水域表示データを簡略化する

water_bodies_display_projected = water_bodies.to_crs(3857)[
    [
        "name",
        "name:en",
        "natural",
        "water",
        "source",
        "geometry",
    ]
].copy()

water_bodies_display_projected["geometry"] = water_bodies_display_projected[
    "geometry"
].simplify(
    WATER_BODY_SIMPLIFICATION_METRES,
    preserve_topology=True,
)


# Repair polygon geometries affected by simplification.
# 簡略化によって無効になった水域Polygonを修復する

invalid_simplified_water_body_count = int(
    (~water_bodies_display_projected["geometry"].is_valid).sum()
)

if invalid_simplified_water_body_count > 0:

    water_bodies_display_projected["geometry"] = water_bodies_display_projected[
        "geometry"
    ].buffer(0)

remaining_invalid_water_body_count = int(
    (~water_bodies_display_projected["geometry"].is_valid).sum()
)

empty_repaired_water_body_count = int(
    water_bodies_display_projected["geometry"].is_empty.sum()
)

if remaining_invalid_water_body_count > 0:

    raise ValueError("Invalid mapped water-body geometries remain " "after repair.")

if empty_repaired_water_body_count > 0:

    raise ValueError(
        "The mapped water-body geometry repair " "produced empty geometries."
    )

water_bodies_web = water_bodies_display_projected.to_crs(4326)

print(
    "Invalid water-body geometries after simplification: "
    f"{invalid_simplified_water_body_count:,}"
)

print(
    "Invalid water-body geometries after repair: "
    f"{remaining_invalid_water_body_count:,}"
)


# Validate all web-display layers.
# Web地図表示用レイヤーを検証する

web_context_datasets = {
    "Country boundary": admin0_web,
    "Governorate boundaries": admin1_web,
    "Rivers": rivers_web,
    "Mapped water bodies": water_bodies_web,
}

for dataset_name, dataset in web_context_datasets.items():

    if dataset.crs.to_epsg() != 4326:

        raise ValueError(f"{dataset_name} was not prepared in EPSG:4326.")

    if dataset["geometry"].isna().any():

        raise ValueError(f"{dataset_name} contains missing display geometries.")

    if dataset["geometry"].is_empty.any():

        raise ValueError(f"{dataset_name} contains empty display geometries.")

    if not dataset["geometry"].is_valid.all():

        raise ValueError(f"{dataset_name} contains invalid display geometries.")

print("Contextual spatial-layer preparation: passed")

for dataset_name, dataset in web_context_datasets.items():

    print(f"{dataset_name}: " f"{len(dataset):,} features, " f"{dataset.crs}")

In [ ]:
# 25
# Create the interactive atlas map and navigation controls
# インタラクティブAtlas地図と操作コントロールを作成する

country_bounds = admin0_web.total_bounds

map_bounds = [
    [
        country_bounds[1],
        country_bounds[0],
    ],
    [
        country_bounds[3],
        country_bounds[2],
    ],
]

m = folium.Map(
    location=[
        35.0,
        38.5,
    ],
    zoom_start=6,
    tiles=None,
    control_scale=True,
    prefer_canvas=True,
)

folium.TileLayer(
    tiles=("https://{s}.basemaps.cartocdn.com/" "light_nolabels/{z}/{x}/{y}{r}.png"),
    attr=("&copy; OpenStreetMap contributors " "&copy; CARTO"),
    name="CARTOライト（地名なし）",
    overlay=False,
    control=True,
    show=True,
).add_to(m)

m.fit_bounds(
    map_bounds,
    padding=[
        20,
        20,
    ],
)

Fullscreen(
    position="topleft",
    title="全画面表示",
    title_cancel="全画面表示を終了",
    force_separate_button=True,
).add_to(m)

mini_map_tiles = folium.TileLayer(
    tiles=("https://{s}.basemaps.cartocdn.com/" "light_nolabels/{z}/{x}/{y}{r}.png"),
    attr=("&copy; OpenStreetMap contributors " "&copy; CARTO"),
)

MiniMap(
    tile_layer=mini_map_tiles,
    position="bottomleft",
    width=180,
    height=120,
    collapsed_width=24,
    collapsed_height=24,
    zoom_level_offset=-4,
    toggle_display=True,
).add_to(m)

print("Interactive atlas map: created")

print(f"Initial map bounds: {map_bounds}")

In [ ]:
# 26
# Add the country and governorate boundary layers
# 国境および県境レイヤーを追加する

country_boundary_group = folium.FeatureGroup(
    name="シリア国境",
    overlay=True,
    control=True,
    show=True,
)

folium.GeoJson(
    data=admin0_web,
    name="シリア国境",
    style_function=lambda feature: {
        "color": "#222222",
        "weight": 2.8,
        "fillColor": "#FFFFFF",
        "fillOpacity": 0.02,
    },
    highlight_function=lambda feature: {
        "color": "#111111",
        "weight": 3.4,
        "fillOpacity": 0.04,
    },
    tooltip=folium.GeoJsonTooltip(
        fields=[
            "adm0_name",
            "adm0_pcode",
        ],
        aliases=[
            "国:",
            "Pcode:",
        ],
        sticky=False,
        labels=True,
    ),
).add_to(country_boundary_group)

country_boundary_group.add_to(m)


governorate_boundary_group = folium.FeatureGroup(
    name="県境",
    overlay=True,
    control=True,
    show=True,
)

folium.GeoJson(
    data=admin1_web,
    name="県境",
    style_function=lambda feature: {
        "color": "#666666",
        "weight": 1.1,
        "fillColor": "#FFFFFF",
        "fillOpacity": 0.01,
    },
    highlight_function=lambda feature: {
        "color": "#333333",
        "weight": 2.0,
        "fillOpacity": 0.05,
    },
    tooltip=folium.GeoJsonTooltip(
        fields=[
            "adm1_name",
            "adm1_pcode",
        ],
        aliases=[
            "県：",
            "Pcode：",
        ],
        sticky=False,
        labels=True,
    ),
).add_to(governorate_boundary_group)

governorate_boundary_group.add_to(m)

print("Administrative boundary layers: added")

print(f"Country features: {len(admin0_web):,}")

print(f"Governorate features: {len(admin1_web):,}")

In [ ]:
# 27
# Add governorate and neighbouring-country labels
# 県名および周辺国名のラベルを追加する

governorate_label_group = folium.FeatureGroup(
    name="県名",
    overlay=True,
    control=True,
    show=True,
)

for governorate in admin1_web.itertuples(index=False):

    if pd.isna(governorate.center_lat) or pd.isna(governorate.center_lon):
        continue

    governorate_name = escape(str(governorate.adm1_name))

    folium.Marker(
        location=[
            float(governorate.center_lat),
            float(governorate.center_lon),
        ],
        icon=folium.DivIcon(
            icon_size=(
                150,
                24,
            ),
            icon_anchor=(
                75,
                12,
            ),
            html=f"""
                <div style="
                    width: 150px;
                    text-align: center;
                    color: #333333;
                    font-size: 12px;
                    font-weight: 700;
                    text-shadow:
                        -1px -1px 0 #FFFFFF,
                         1px -1px 0 #FFFFFF,
                        -1px  1px 0 #FFFFFF,
                         1px  1px 0 #FFFFFF;
                    pointer-events: none;
                ">
                    {governorate_name}
                </div>
            """,
        ),
    ).add_to(governorate_label_group)

governorate_label_group.add_to(m)


neighbour_label_locations = {
    "TÜRKİYE": (
        37.55,
        37.60,
    ),
    "IRAQ": (
        34.55,
        43.10,
    ),
    "LEBANON": (
        33.75,
        35.55,
    ),
    "JORDAN": (
        31.95,
        36.85,
    ),
}

neighbour_label_group = folium.FeatureGroup(
    name="周辺国名",
    overlay=True,
    control=True,
    show=True,
)

for neighbour_name, coordinates in neighbour_label_locations.items():

    folium.Marker(
        location=coordinates,
        icon=folium.DivIcon(
            icon_size=(
                160,
                30,
            ),
            icon_anchor=(
                80,
                15,
            ),
            html=f"""
                <div style="
                    width: 160px;
                    text-align: center;
                    color: #666666;
                    font-size: 17px;
                    font-weight: 800;
                    text-shadow:
                        -1px -1px 0 #FFFFFF,
                         1px -1px 0 #FFFFFF,
                        -1px  1px 0 #FFFFFF,
                         1px  1px 0 #FFFFFF;
                    pointer-events: none;
                ">
                    {escape(neighbour_name)}
                </div>
            """,
        ),
    ).add_to(neighbour_label_group)

neighbour_label_group.add_to(m)

print("Map label layers: added")

print(f"Governorate labels: {len(admin1_web):,}")

print("Neighbour labels: " f"{len(neighbour_label_locations):,}")

In [ ]:
# 28
# Define the atlas popup and proportional-layer functions
# Atlasのポップアップおよび比例円レイヤー関数を定義する


def safe_popup_text(value):

    if pd.isna(value):
        return "Not recorded"

    text_value = str(value).strip()

    if not text_value:
        return "Not recorded"

    return escape(text_value)


def build_atlas_popup(
    record,
    selected_indicator_label,
    selected_indicator_name,
):

    selected_value = int(
        getattr(
            record,
            selected_indicator_name,
        )
    )

    location_name = safe_popup_text(record.location_name)

    adm1_name = safe_popup_text(record.adm1_name)

    adm2_name = safe_popup_text(record.adm2_name)

    adm3_name = safe_popup_text(record.adm3_name)

    location_pcode = safe_popup_text(record.location_pcode)

    hot_matching_name = safe_popup_text(record.hot_matching_name)

    match_status = safe_popup_text(record.match_status)

    popup_html = f"""
        <div style="
            width: 310px;
            font-family: Arial, sans-serif;
            color: #222222;
        ">
            <div style="
                color: #1D3557;
                font-size: 17px;
                font-weight: 700;
                margin-bottom: 7px;
                padding-bottom: 6px;
                border-bottom: 1px solid #DDDDDD;
            ">
                {location_name}
            </div>

            <div style="
                font-size: 13px;
                line-height: 1.45;
            ">
                <b>{escape(selected_indicator_label)}:</b>
                {selected_value:,}<br>

                <b>Governorate:</b>
                {adm1_name}<br>

                <b>District:</b>
                {adm2_name}<br>

                <b>Sub-District:</b>
                {adm3_name}<br>

                <b>Location Pcode:</b>
                {location_pcode}<br>

                <hr style="
                    border: 0;
                    border-top: 1px solid #DDDDDD;
                    margin: 7px 0;
                ">

                <b>Residents:</b>
                {int(record.residents):,}<br>

                <b>IDPs:</b>
                {int(record.idps):,}<br>

                <b>IDP Returnees Since December 2024:</b>
                {int(record.idp_returnees_dec2024):,}<br>

                <b>IDP Returnees in 2026:</b>
                {int(record.idp_returnees_2026):,}<br>

                <b>Total Population:</b>
                {int(record.total_population):,}<br>

                <hr style="
                    border: 0;
                    border-top: 1px solid #DDDDDD;
                    margin: 7px 0;
                ">

                <b>Matched HOTOSM name:</b>
                {hot_matching_name}<br>

                <b>Match status:</b>
                {match_status}<br>

                <b>Similarity score:</b>
                {float(record.similarity_score):.2f}
            </div>
        </div>
    """

    return popup_html


def add_population_atlas_layer(
    map_object,
    layer_name,
    layer_definition,
    show=False,
):

    layer_data = layer_definition["data"]

    indicator_name = layer_definition["indicator"]

    layer_colour = layer_definition["colour"]

    feature_group = folium.FeatureGroup(
        name=layer_name,
        overlay=True,
        control=True,
        show=show,
    )

    for record in layer_data.itertuples(index=False):

        indicator_value = int(
            getattr(
                record,
                indicator_name,
            )
        )

        circle_radius = calculate_circle_radius(indicator_value)

        tooltip_text = (
            f"{layer_name} | " f"{record.location_name} | " f"{indicator_value:,}"
        )

        popup_html = build_atlas_popup(
            record,
            layer_name,
            indicator_name,
        )

        folium.CircleMarker(
            location=[
                float(record.geometry.y),
                float(record.geometry.x),
            ],
            radius=circle_radius,
            color=layer_colour,
            weight=1.0,
            fill=True,
            fill_color=layer_colour,
            fill_opacity=0.55,
            tooltip=tooltip_text,
            popup=folium.Popup(
                popup_html,
                max_width=350,
            ),
        ).add_to(feature_group)

    feature_group.add_to(map_object)

    return feature_group


# Add a reference layer containing all validated locations.
# 検証済み地点すべてを参照レイヤーとして追加する

validated_location_group = folium.FeatureGroup(
    name=("検証済み照合地点 " f"({len(validated_locations):,})"),
    overlay=True,
    control=True,
    show=False,
)

for record in validated_locations.itertuples(index=False):

    popup_html = build_atlas_popup(
        record,
        "Total Population",
        "total_population",
    )

    folium.CircleMarker(
        location=[
            float(record.geometry.y),
            float(record.geometry.x),
        ],
        radius=2.0,
        color="#555555",
        weight=0.8,
        fill=True,
        fill_color="#FFFFFF",
        fill_opacity=0.75,
        tooltip=(f"{record.location_name} | " f"{record.location_pcode}"),
        popup=folium.Popup(
            popup_html,
            max_width=350,
        ),
    ).add_to(validated_location_group)

validated_location_group.add_to(m)

print("Atlas helper functions: defined")

print(
    "Validated matched-location reference layer: "
    f"{len(validated_locations):,} points"
)

In [ ]:
# 29
# Add the residents proportional-circle layer
# 居住者数の比例円レイヤーを追加する

residents_feature_group = add_population_atlas_layer(
    map_object=m,
    layer_name="居住者",
    layer_definition=atlas_layers["Residents"],
    show=False,
)

print("Residents layer added: " f"{len(residents_layer):,} locations")

print("Residents represented: " f"{int(residents_layer['residents'].sum()):,}")

In [ ]:
# 30
# Add the IDP proportional-circle layer
# 国内避難民数の比例円レイヤーを追加する

idps_feature_group = add_population_atlas_layer(
    map_object=m,
    layer_name="国内避難民（IDP）",
    layer_definition=atlas_layers["IDPs"],
    show=True,
)

print("IDP layer added: " f"{len(idps_layer):,} locations")

print("IDPs represented: " f"{int(idps_layer['idps'].sum()):,}")

In [ ]:
# 31
# Add the IDP returnees since December 2024 layer
# 2024年12月以降のIDP帰還者比例円レイヤーを追加する

idp_returnees_dec2024_feature_group = add_population_atlas_layer(
    map_object=m,
    layer_name=("IDP帰還者（2024年12月以降）"),
    layer_definition=atlas_layers["IDP Returnees Since December 2024"],
    show=True,
)

print(
    "IDP returnees since December 2024 layer added: "
    f"{len(idp_returnees_dec2024_layer):,} locations"
)

print(
    "IDP returnees since December 2024 represented: "
    f"{int(idp_returnees_dec2024_layer['idp_returnees_dec2024'].sum()):,}"
)

In [ ]:
# 32
# Add the IDP returnees recorded in 2026 layer
# 2026年に記録されたIDP帰還者比例円レイヤーを追加する

idp_returnees_2026_feature_group = add_population_atlas_layer(
    map_object=m,
    layer_name="IDP帰還者（2026年）",
    layer_definition=atlas_layers["IDP Returnees in 2026"],
    show=False,
)

print(
    "IDP returnees in 2026 layer added: " f"{len(idp_returnees_2026_layer):,} locations"
)

print(
    "IDP returnees in 2026 represented: "
    f"{int(idp_returnees_2026_layer['idp_returnees_2026'].sum()):,}"
)

In [ ]:
# 33
# Add the total-population proportional-circle layer
# 総人口の比例円レイヤーを追加する

total_population_feature_group = add_population_atlas_layer(
    map_object=m,
    layer_name="総人口",
    layer_definition=atlas_layers["Total Population"],
    show=False,
)

print("Total-population layer added: " f"{len(total_population_layer):,} locations")

print(
    "Total population represented: "
    f"{int(total_population_layer['total_population'].sum()):,}"
)

In [ ]:
# 34
# Add the river and mapped water-body context layers
# 河川および水域の背景レイヤーを追加する

mapped_water_body_group = folium.FeatureGroup(
    name="水域データ",
    overlay=True,
    control=True,
    show=True,
)

folium.GeoJson(
    data=water_bodies_web[
        [
            "geometry",
        ]
    ],
    name="Mapped Water Bodies",
    style_function=lambda feature: {
        "color": "#219EBC",
        "weight": 0.7,
        "fillColor": "#8ECAE6",
        "fillOpacity": 0.30,
    },
    highlight_function=lambda feature: {
        "color": "#0077B6",
        "weight": 1.2,
        "fillOpacity": 0.42,
    },
).add_to(mapped_water_body_group)

mapped_water_body_group.add_to(m)


river_feature_group = folium.FeatureGroup(
    name="河川データ",
    overlay=True,
    control=True,
    show=True,
)

folium.GeoJson(
    data=rivers_web[
        [
            "geometry",
        ]
    ],
    name="Mapped Rivers",
    style_function=lambda feature: {
        "color": "#0077B6",
        "weight": 1.1,
        "opacity": 0.75,
    },
    highlight_function=lambda feature: {
        "color": "#023E8A",
        "weight": 2.0,
        "opacity": 1.0,
    },
).add_to(river_feature_group)

river_feature_group.add_to(m)


# Keep population symbols above the contextual layers.
# 人口シンボルを背景レイヤーより前面に維持する

m.keep_in_front(
    validated_location_group,
    residents_feature_group,
    idps_feature_group,
    idp_returnees_dec2024_feature_group,
    idp_returnees_2026_feature_group,
    total_population_feature_group,
)

print("Hydrography context layers: added")

print(f"Mapped river features: {len(rivers_web):,}")

print("Mapped water-body features: " f"{len(water_bodies_web):,}")

In [ ]:
# 35
# Add the information panel, shared legend and layer control
# 情報パネル、共通凡例およびレイヤー操作を追加する

accepted_location_count = len(validated_locations)

source_location_count = len(iom)

unresolved_location_count = source_location_count - accepted_location_count

accepted_location_percentage = accepted_location_count / source_location_count * 100

exact_match_count = int(validated_locations["match_status"].eq("Exact").sum())

high_confidence_match_count = int(
    validated_locations["match_status"].eq("High confidence").sum()
)


# Prepare the information panel.
# 地図説明パネルを準備する

information_panel_html = f"""
<div style="
    position: fixed;
    top: 20px;
    left: 58px;
    width: 425px;
    max-height: 520px;
    overflow-y: auto;
    background-color: rgba(255, 255, 255, 0.95);
    color: #222222;
    z-index: 9000;
    font-family:
        'Hiragino Sans',
        'Yu Gothic',
        Arial,
        sans-serif;
    font-size: 13px;
    line-height: 1.38;
    border: 1px solid #555555;
    border-radius: 8px;
    padding: 13px;
    box-shadow: 0 0 15px rgba(0, 0, 0, 0.24);
">
    <div style="
        font-size: 18px;
        font-weight: 700;
        margin-bottom: 2px;
    ">
        Syria
    </div>

    <div style="
        color: #1D3557;
        font-size: 15px;
        font-weight: 700;
        margin-bottom: 8px;
    ">
        Displacement and Return Atlas (2026)
    </div>

    <div>
        このアトラスでは、国際移住機関（IOM）の避難・帰還・人口指標を、同じADM3（第3行政区分）内にあるHumanitarian OpenStreetMap Team（HOTOSM）の集落ポイントと照合しています。地図上には、完全一致または高信頼度と判定された一対一の照合結果のみを表示しています。
    </div>

    <div style="
        margin-top: 9px;
        padding-top: 8px;
        border-top: 1px solid #AAAAAA;
    ">
        <b>IOM元データ地点数：</b>
        {source_location_count:,}<br>

        <b>検証済み地図表示地点数：</b>
        {accepted_location_count:,}
        ({accepted_location_percentage:.2f}%)<br>

        <b>完全一致：</b>
        {exact_match_count:,}<br>

        <b>高信頼度一致：</b>
        {high_confidence_match_count:,}<br>

        <b>未確定地点数：</b>
        {unresolved_location_count:,}<br>

        <b>自動採用基準：</b>
        {HIGH_CONFIDENCE_THRESHOLD:.0f}/100
    </div>

    <div style="
        margin-top: 9px;
        padding-top: 8px;
        border-top: 1px solid #AAAAAA;
        color: #555555;
        font-size: 11px;
    ">
        <b>Population data:</b>
        IOM displacement and return dataset, 2026<br>

        <b>Settlement geometry:</b>
        HOTOSM populated places<br>

        <b>Method:</b>
        ADM3-constrained name matching /
        one-to-one assignment /
        geometry validation<br>

        表示される合計値は、検証済みの地図表示地点のみを対象としており、IOM元データの全レコードを表すものではありません。
    </div>
</div>
"""

m.get_root().html.add_child(Element(information_panel_html))

# Prepare common proportional-circle legend values.
# 共通比例円凡例の値を準備する

legend_values = [
    circle_reference_value,
    circle_reference_value * 0.50,
    circle_reference_value * 0.25,
]

legend_rows = ""

for legend_value in legend_values:

    legend_radius = calculate_circle_radius(legend_value)

    legend_diameter = legend_radius * 2

    legend_rows += f"""
        <div style="
            display: flex;
            align-items: center;
            min-height: 52px;
            margin-top: 3px;
        ">
            <div style="
                width: 56px;
                display: flex;
                justify-content: center;
                align-items: center;
            ">
                <div style="
                    width: {legend_diameter:.1f}px;
                    height: {legend_diameter:.1f}px;
                    border-radius: 50%;
                    background-color: rgba(244, 162, 97, 0.55);
                    border: 1px solid #F4A261;
                    box-sizing: border-box;
                "></div>
            </div>

            <div style="
                margin-left: 8px;
                font-size: 12px;
            ">
                {legend_value:,.0f}
            </div>
        </div>
    """

legend_html = f"""
<div style="
    position: fixed;
    right: 25px;
    bottom: 25px;
    width: 265px;
    background-color: rgba(255, 255, 255, 0.95);
    color: #222222;
    z-index: 9000;
    font-family:
        'Hiragino Sans',
        'Yu Gothic',
        Arial,
        sans-serif;
    border: 1px solid #555555;
    border-radius: 8px;
    padding: 12px;
    box-shadow: 0 0 12px rgba(0, 0, 0, 0.22);
">
    <div style="
        font-size: 15px;
        font-weight: 700;
        margin-bottom: 6px;
    ">
        人口指標共通の比例円スケール
    </div>

    {legend_rows}

    <div style="
        margin-top: 7px;
        padding-top: 7px;
        border-top: 1px solid #AAAAAA;
        font-size: 11px;
        line-height: 1.35;
        color: #555555;
    ">
        円の大きさは、各地点の人数に応じて変化します。5種類の人口データには共通の基準を使用しているため、円の大きさを相互に比較できます。極端に大きな値は、地図上での重なりを抑えるため最大サイズで表示しています。
    </div>

    <div style="
        margin-top: 8px;
        font-size: 11px;
        line-height: 1.45;
    ">
        <span style="color: #2A9D8F;">●</span>
        居住者<br>

        <span style="color: #E76F51;">●</span>
        国内避難民（IDP）<br>

        <span style="color: #d1908d;">●</span>
        IDP帰還者（2024年12月以降）<br>

        <span style="color: #8B1E3F;">●</span>
        IDP帰還者（2026年）<br>

        <span style="color: #F4A261;">●</span>
        総人口
    </div>
</div>
"""

m.get_root().html.add_child(Element(legend_html))

folium.LayerControl(
    position="topright",
    collapsed=False,
    autoZIndex=True,
).add_to(m)

print("Atlas information panel: added")

print("Shared proportional-circle legend: added")

print("Layer control: added")

In [ ]:
# 36
# Save and display the interactive map
# インタラクティブ地図を保存し、Notebook上に表示する

m.save(output_path)

if not output_path.exists():

    raise FileNotFoundError("The interactive atlas HTML file was not created.")

output_file_size_mb = output_path.stat().st_size / (1024 * 1024)

print(f"Map saved to: {output_path}")

print("HTML file size: " f"{output_file_size_mb:.2f} MB")

m